# SpectraShift Week 8: seed 43 final evaluation
Use T4 x2 with Internet off and GPU 0. Attach source v7, Week 2 frozen, Week 6 contracts, Week 7 contracts v2, Week 7 complete, Week 8 evaluation contracts, and matching Week 5/6/7 seed datasets.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, yaml

INPUT = Path('/kaggle/input')
projects = [p.parent for p in INPUT.rglob('pyproject.toml') if (p.parent / 'src/spectrashift/train/week8.py').is_file()]
if not projects:
    bundles = sorted(INPUT.rglob('spectrashift-kaggle-source.zip'))
    assert len(bundles) == 1, f'Expected one Week 8 source bundle, found {bundles}'
    source_work = Path('/tmp/spectrashift-week8-source')
    if source_work.exists(): shutil.rmtree(source_work)
    shutil.unpack_archive(str(bundles[0]), str(source_work))
    projects = [source_work]
assert projects, 'No Week 8 source tree found'
PROJECT = sorted(projects, key=lambda path: len(str(path)))[0]
sys.path.insert(0, str(PROJECT / 'src'))
os.chdir(PROJECT)

def unique_file(name):
    candidates = sorted(INPUT.rglob(name))
    by_hash = {}
    for path in candidates:
        by_hash.setdefault(hashlib.sha256(path.read_bytes()).hexdigest(), path)
    assert len(by_hash) == 1, f'Expected one unique {name}; found {candidates}'
    return next(iter(by_hash.values()))

def install_offline_foundation_dependencies():
    wheels = sorted(INPUT.rglob('foundation-wheels'))
    if not wheels: return
    missing = []
    for module, package in [('upath','universal-pathlib'), ('omegaconf','omegaconf'), ('iopath','iopath'), ('fvcore','fvcore'), ('einops','einops'), ('huggingface_hub','huggingface_hub')]:
        try: __import__(module)
        except ImportError: missing.append(package)
    if missing:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-index', '--find-links', str(wheels[0]), *missing])

install_offline_foundation_dependencies()
SEED = 43
WORK = Path('/kaggle/working/spectrashift-week8-seed43-eval')
WORK.mkdir(parents=True, exist_ok=True)
MANIFEST = unique_file('partitions.parquet')
NORMALIZATION = unique_file('normalization.json')
FREEZE = unique_file('freeze_summary.json')
WEEK7_SUMMARY = unique_file('week7_run_summary.json')
LEDGER = unique_file('week8_checkpoint_ledger.csv')
STAGED = next(path.parent for path in INPUT.rglob('staging_summary.json'))
RGB_CONTRACT = unique_file('rgb_percentile_contract.json')
WEEK7_CONTRACTS = unique_file('week7_contracts_summary.json')
OLMO_CONTRACT = unique_file('olmoearth_input_contract.json')
EVAL_LABELS = unique_file('evaluation_labels.parquet')
EVAL_CONTRACT = unique_file('evaluation_contract.json')
SUPPORT_CONTRACT = unique_file('support_contract.json')
config = yaml.safe_load((PROJECT / 'configs/eval/week8.yaml').read_text())
config['paths'].update({
    'manifest_path': str(MANIFEST), 'staged_root': str(STAGED),
    'normalization_path': str(NORMALIZATION), 'freeze_summary_path': str(FREEZE),
    'week7_summary_path': str(WEEK7_SUMMARY), 'checkpoint_ledger_path': str(LEDGER),
    'evaluation_labels_path': str(EVAL_LABELS), 'evaluation_contract_path': str(EVAL_CONTRACT),
    'support_contract_path': str(SUPPORT_CONTRACT), 'rgb_contract_path': str(RGB_CONTRACT),
    'week7_contracts_path': str(WEEK7_CONTRACTS), 'olmo_contract_path': str(OLMO_CONTRACT),
})
RUNTIME_CONFIG = WORK / 'week8.yaml'
RUNTIME_CONFIG.write_text(yaml.safe_dump(config, sort_keys=False))
print({'seed': SEED, 'project': str(PROJECT), 'work': str(WORK)})


In [ ]:
import torch
assert torch.cuda.is_available() and 'T4' in torch.cuda.get_device_name(0), 'Select GPU T4 x2'
from spectrashift.train.week8 import evaluate_week8_seed
summary = evaluate_week8_seed(RUNTIME_CONFIG, SEED, [INPUT], WORK)
print(json.dumps({key: value for key, value in summary.items() if key != 'runs'}, indent=2))
assert summary['week8_seed_evaluation_complete']
assert summary['run_count'] == 37 and summary['prediction_domain_count'] == 111
assert summary['model_selection_after_label_access'] is False
assert summary['parameter_updates_after_label_access'] is False
